# JeleBondhu · Colab Pro calibration notebook

Builds the **optional reference** for anomaly variables. It does NOT, by itself, upgrade Filter 1 to validated frozen thesis mode and refreshes the
event-grouped validation. Nothing here runs on the public site; outputs are small files you commit to the GitHub repo.

| Output | Used by | Effect |
|---|---|---|
| `data/reference/climatology.npz` | `backend/jelebondhu/reference.py` | monthly SST / sea-level climatology at the 2,125 stations → anomaly components (`sst_anom`, `zos_anom`) |
| `data/static/validation_summary.json` | web app **Evidence** tab | event-grouped agreement check vs in-situ data |

**Security:** never type your Copernicus password into a notebook cell. This notebook uses `getpass`, which keeps it out of the saved `.ipynb`. The password you pasted into chat earlier should be **rotated** at https://data.marine.copernicus.eu → account settings.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, getpass, subprocess
REPO_URL = input('GitHub repo URL (https://github.com/<you>/<repo>.git): ').strip()
if not os.path.exists('/content/jelebondhu'):
    url = REPO_URL  # public repository: never inject a PAT into a clone URL
    subprocess.run(['git','clone',url,'/content/jelebondhu'],check=True)
os.chdir('/content/jelebondhu')
!pip -q install numpy pandas openpyxl xarray netCDF4 h5netcdf copernicusmarine pyarrow

## A · Monthly climatology from the Copernicus reanalysis (recommended)

Uses the GLORYS12 monthly reanalysis. **Check the dataset ID** in the Copernicus Marine catalogue before running
(IDs are occasionally renamed). Processing 1995–2025 requires time, network access and careful memory management; subset by region and period.

In [ ]:
import numpy as np, json, copernicusmarine as cm, xarray as xr
user = input('CMEMS username: ').strip(); pwd = getpass.getpass('CMEMS password: ')
cm.login(username=user, password=pwd, force_overwrite=True)
DATASET = 'cmems_mod_glo_phy_my_0.083deg_P1M-m'          # <- verify in catalogue
st = json.load(open('data/static/stations.json')); lat = np.array(st['lat']); lon = np.array(st['lon']); n = st['n']
ds = cm.open_dataset(dataset_id=DATASET, variables=['thetao','zos'], minimum_longitude=88.0, maximum_longitude=93.2,
                     minimum_latitude=17.2, maximum_latitude=23.3, minimum_depth=0, maximum_depth=1,
                     start_datetime='1995-01-01', end_datetime='2025-12-31')
sst = ds['thetao'].isel(depth=0); zos = ds['zos']
print(ds)

In [ ]:
import pandas as pd
rows = []
la = xr.DataArray(lat, dims='s'); lo = xr.DataArray(lon, dims='s')
for name, da in (('sst', sst), ('zos', zos)):
    clim = da.groupby('time.month').mean('time')                       # 12 x lat x lon
    pts = clim.interp(latitude=la, longitude=lo, method='nearest')   # nearest valid cell at each station
    rows.append(pd.DataFrame(pts.values.T, columns=pts['month'].values).assign(station_idx=range(n)).melt('station_idx', var_name='month', value_name=name))
df = rows[0].merge(rows[1], on=['station_idx','month'])
import sys; sys.path.insert(0, 'backend')
from jelebondhu.reference import build_reference
os.makedirs('data/reference', exist_ok=True)
print(build_reference(df, n, 'data/reference/climatology.npz'))
print('NaN share per variable:', df[['sst','zos']].isna().mean().to_dict())

## B · (alternative) climatology from your own 1995–2025 archive

Export the Google Sheet to Parquet/CSV on Drive first (a 1.35 GB sheet is too slow to read live). Required columns:
`lat`, `lon`, `month`, and `sst` and/or `zos` (rename below). Rows are mapped to the nearest station.

In [ ]:
# df_arch = pd.read_parquet('/content/drive/MyDrive/JeleBondhu/archive_1995_2025.parquet', columns=['lat','lon','month','sst','zos'])
# from scipy.spatial import cKDTree
# tree = cKDTree(np.c_[lat, lon]); d, idx = tree.query(np.c_[df_arch.lat, df_arch.lon])
# df_arch = df_arch[d < 0.1].assign(station_idx=idx[d < 0.1])
# print(build_reference(df_arch, n, 'data/reference/climatology.npz'))

## C · Save outputs and commit

In [ ]:
!cp data/reference/climatology.npz /content/drive/MyDrive/JeleBondhu/climatology.npz 2>/dev/null || true
print('Now: git add data/reference/climatology.npz && git commit -m "Add reference climatology" && git push')
print('Then set  "mode": "frozen"  in config/filter1_environment.json ONLY after the frozen thesis membership functions are supplied.')

## D · Event-grouped validation vs in-situ fisheries data

Upload `JeleBondhu_Commercial_Fisheries_PlotReady_EEZ_*.xlsx` to Drive. The script aggregates to **one row per event (DLC_ID)**,
uses a spatial-block bootstrap, and reports data-quality diagnostics. Read its findings before quoting any skill.

In [ ]:
XLSX = '/content/drive/MyDrive/JeleBondhu/JeleBondhu_Commercial_Fisheries_PlotReady_EEZ_20261124.xlsx'
!python scripts/validate_events.py --xlsx "$XLSX"